# 🏛️ MPLADS AI Platform: Official Data Ingestion & EDA
**Smart India Hackathon 2026 — Problem Statement:** `SIH26102`  
**Ministry:** Ministry of Statistics and Programme Implementation (MoSPI) / DIID  

This notebook demonstrates:
1. **Live REST Ingestion** from MoSPI e-SAKSHI Portal (`https://mplads.mospi.gov.in/digigov/dashboard.html`)
2. **Data Cleaning & Standardization** (Currency parsing, date sequence checking, surrogate key generation)
3. **Exploratory Data Analysis** (Sectoral allocations, state expenditures, cost distributions)

## 1. Environment Setup & Ingestion Pipeline

In [1]:
import sys
from pathlib import Path
# Add project root to sys.path
sys.path.append(str(Path("..").resolve()))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Setup complete. Python libraries loaded.")

## 2. Load Official e-SAKSHI Datasets & Master Work Lake

In [2]:
# Load processed master works dataset
master_path = Path("../data/processed/master_works.parquet")
df = pd.read_parquet(master_path)

print(f"📊 Master Dataset Loaded: {len(df):,} works across {df['state'].nunique()} Indian States/UTs.")
print(f"💰 Total Sanctioned Budget: ₹{df['sanctioned_amount'].sum()/1e7:,.2f} Crore")
print(f"💸 Total Expenditure: ₹{df['total_expenditure'].sum()/1e7:,.2f} Crore")
df.head()

## 3. Exploratory Data Analysis & Visualizations

In [3]:
# 1. Sectoral Distribution
sector_counts = df['sector'].value_counts()

plt.figure(figsize=(12, 5))
sns.barplot(x=sector_counts.values, y=sector_counts.index, palette="Blues_r")
plt.title("Distribution of MPLADS Infrastructure Works by Sector", fontsize=13, fontweight="bold")
plt.xlabel("Number of Works")
plt.ylabel("Sector")
plt.tight_layout()
plt.show()

In [4]:
# 2. Sanctioned Cost vs. Actual Expenditure by State
state_summary = df.groupby('state').agg(
    sanctioned=('sanctioned_amount', lambda x: x.sum()/1e7),
    expenditure=('total_expenditure', lambda x: x.sum()/1e7)
).sort_values('sanctioned', ascending=False).head(10)

state_summary.plot(kind='bar', figsize=(12, 6), color=['#2563eb', '#10b981'])
plt.title("Top 10 States: Total Sanctioned Budget vs Actual Expenditure (₹ Crore)", fontsize=13, fontweight="bold")
plt.ylabel("Amount (₹ Crore)")
plt.xticks(rotation=45, ha='right')
plt.legend(["Sanctioned Budget", "Actual Expenditure"])
plt.tight_layout()
plt.show()